# CTI classifier improvement: leakage-aware TF-IDF, SBERT, and Hybrid

This notebook improves evaluation rather than promising a particular accuracy. It uses **group-aware cross-validation** and fits TF-IDF separately inside each CV fold. It assumes Notebook 02 has already created `artifacts/train.csv` and `artifacts/test.csv` with `document_id`, `sentence_text`, and binary `relevant` (0/1). Run from the project `notebook` directory.

**Important:** keep the test set untouched until model selection is complete. The reported scores may differ from earlier results because the evaluation protocol is stricter.

## 1. Imports, paths, and validation

In [1]:
from pathlib import Path
import json, time, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)
from scipy.sparse import hstack
from sklearn.preprocessing import normalize

cwd = Path.cwd()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "artifacts" / "train.csv").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find artifacts/train.csv. Run Notebook 02 first and start from the notebook directory.")
ART = PROJECT_ROOT / "artifacts"
(ART / "figs").mkdir(parents=True, exist_ok=True)
train = pd.read_csv(ART / "train.csv")
test = pd.read_csv(ART / "test.csv")
required = {"sentence_text", "relevant", "document_id"}
if not required.issubset(train.columns) or not required.issubset(test.columns):
    raise ValueError(f"train/test must contain {sorted(required)}")
for name, part in [("train", train), ("test", test)]:
    part["sentence_text"] = part["sentence_text"].fillna("").astype(str)
    part["relevant"] = pd.to_numeric(part["relevant"], errors="raise").astype(int)
    if set(part["relevant"].unique()) != {0, 1}:
        raise ValueError(f"{name} must contain both binary labels 0 and 1")

print("Train:", train.shape, "Test:", test.shape)
print("Train label proportions:
", train.relevant.value_counts(normalize=True).sort_index().round(3))
print("Test label proportions:
", test.relevant.value_counts(normalize=True).sort_index().round(3))
print("Train documents:", train.document_id.nunique(), "Test documents:", test.document_id.nunique())
print("Exact text overlap train/test:", len(set(train.sentence_text) & set(test.sentence_text)))


SyntaxError: unterminated string literal (detected at line 37) (4095057594.py, line 37)

## 2. Build groups for leakage-aware cross-validation

Documents that share an exact sentence are linked into one connected component. All rows in a component stay in the same CV fold. This reduces the chance that repeated text appears in both the training and validation fold. If the number of components/classes is too small, reduce `N_SPLITS` only after checking the data.

In [ ]:
# Union-find over documents sharing exact sentence text
parent = {}
def find(x):
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for doc in train.document_id.astype(str).unique():
    find(doc)
for _, rows in train.groupby("sentence_text", sort=False):
    docs = rows.document_id.astype(str).unique().tolist()
    for doc in docs[1:]:
        union(docs[0], doc)

train_docs = train.document_id.astype(str)
groups = train_docs.map(find).to_numpy()
print("CV connected components:", pd.Series(groups).nunique())
print("Largest component rows:", pd.Series(groups).value_counts().max())

N_SPLITS = 5
if pd.Series(groups).nunique() < N_SPLITS:
    raise ValueError(f"Only {pd.Series(groups).nunique()} connected components; cannot use {N_SPLITS}-fold grouped CV.")
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
# Verify that every fold has no group overlap and both classes
for fold, (tr_i, va_i) in enumerate(cv.split(train, train.relevant, groups), start=1):
    overlap = set(groups[tr_i]) & set(groups[va_i])
    if overlap:
        raise AssertionError(f"Group leakage in fold {fold}")
    print(f"Fold {fold}: train={len(tr_i)}, validation={len(va_i)}, val positive rate={train.relevant.iloc[va_i].mean():.3f}")


## 3. Tune TF-IDF + linear classifiers (vectorizer inside CV)

Unlike fitting TF-IDF once before `GridSearchCV`, this pipeline learns vocabulary and IDF using only the training portion of each fold.

In [ ]:
tfidf_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True, strip_accents="unicode", lowercase=True)),
    ("clf", LinearSVC(random_state=42, max_iter=20000))
])
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_features": [20000, 50000, None],
    "clf__C": [0.1, 0.3, 1.0, 3.0, 10.0],
    "clf__class_weight": [None, "balanced"],
}
# If runtime is excessive, reduce the grid after the first run and document the change.
search_tfidf = GridSearchCV(
    tfidf_pipe, param_grid, scoring="f1_macro", cv=cv,
    n_jobs=-1, refit=True, verbose=1, return_train_score=True
)
t0 = time.time()
search_tfidf.fit(train.sentence_text, train.relevant, groups=groups)
print(f"Search time: {(time.time()-t0)/60:.1f} min")
print("Best grouped-CV macro-F1:", round(search_tfidf.best_score_, 4))
print("Best parameters:", search_tfidf.best_params_)
cv_results = pd.DataFrame(search_tfidf.cv_results_).sort_values("rank_test_score")
cv_results[["params", "mean_test_score", "std_test_score", "mean_train_score", "rank_test_score"]].head(15).to_csv(ART / "grid_tfidf_group_cv.csv", index=False)
display(cv_results[["params", "mean_test_score", "std_test_score", "mean_train_score", "rank_test_score"]].head(10))


## 4. Evaluate selected TF-IDF model once on the held-out test set

In [ ]:
best_tfidf = search_tfidf.best_estimator_
pred_tfidf = best_tfidf.predict(test.sentence_text)
y_test = test.relevant.to_numpy()

def metric_dict(name, y_true, y_pred, cv_score=None):
    result = {
        "model": name,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }
    if cv_score is not None:
        result["cv_macro_f1"] = float(cv_score)
    return result

m_tfidf = metric_dict("TF-IDF + LinearSVC (grouped CV)", y_test, pred_tfidf, search_tfidf.best_score_)
m_tfidf["selection_strategy"] = "StratifiedGroupKFold over document-sentence connected components; TF-IDF fitted inside each fold"
m_tfidf["hyperparams"] = search_tfidf.best_params_
print(json.dumps(m_tfidf, indent=2))
print("\nTest classification report:\n", classification_report(y_test, pred_tfidf, digits=4, zero_division=0))
with open(ART / "metrics_tfidf_group_cv.json", "w", encoding="utf-8") as f:
    json.dump(m_tfidf, f, indent=2)
cm = confusion_matrix(y_test, pred_tfidf, labels=[0,1])
ConfusionMatrixDisplay(cm, display_labels=["non-relevant", "relevant"]).plot(values_format="d")
plt.title("TF-IDF + LinearSVC — held-out test")
plt.tight_layout(); plt.savefig(ART / "figs" / "fig_cm_tfidf_group_cv.png", dpi=160); plt.show()


## 5. SBERT and Hybrid: recommended evaluation corrections

Use the same `groups` and `cv` object in Notebook 04. The following changes are important:

1. Replace `StratifiedKFold` with `StratifiedGroupKFold` and pass `groups=groups` to each `GridSearchCV.fit(...)`.
2. Check the connected-component grouping is derived from the **train split only**.
3. Do not reuse cached embeddings unless row order and text content match.
4. For Hybrid CV, fit TF-IDF inside each fold. Fitting TF-IDF once on all training rows before CV allows validation-fold text to influence vocabulary/IDF.
5. Compare all models using the same held-out test split and report macro-F1, relevant-class recall/precision, and confusion matrix alongside accuracy.

The next cell writes the CV grouping array and a reproducibility record so Notebook 04 can reuse them without silently recreating a different split.

In [ ]:
# Save row-aligned CV group IDs. These correspond to current artifacts/train.csv row order.
np.save(ART / "train_cv_groups.npy", groups)
cv_info = {
    "cv_method": "StratifiedGroupKFold",
    "n_splits": N_SPLITS,
    "random_state": 42,
    "grouping": "connected components of documents linked by exact sentence text within train split",
    "train_rows": int(len(train)),
    "train_components": int(pd.Series(groups).nunique()),
    "tfidf_fitted_inside_cv": True,
    "selection_metric": "macro_f1",
}
with open(ART / "group_cv_info.json", "w", encoding="utf-8") as f:
    json.dump(cv_info, f, indent=2)
print(json.dumps(cv_info, indent=2))
print("Saved metrics:", ART / "metrics_tfidf_group_cv.json")
print("Saved CV groups:", ART / "train_cv_groups.npy")


## 6. Interpretation

Do not assume the score will increase: a leakage-aware evaluation can be lower than an earlier score, and that is a more credible estimate of generalization. Choose the final model using grouped-CV macro-F1, then report its held-out test metrics. Never change the test set repeatedly to chase a higher score. If relevant-class recall is the operational priority, define that objective before tuning and use an appropriate scorer rather than selecting whichever metric looks best afterward.